# PolLens: Detection v4 on Google Colab

This notebook loads the **unaugmented Detection v4** export from private Google Drive, clones the version-controlled PolLens training code from GitHub, stages images on Colab's local disk, and saves checkpoints and metrics to Drive. It compares COCO-pretrained YOLO11n and YOLO11s with the same exported split and settings.

**Before running:** upload the complete `Detection-2` folder or a ZIP of it to Drive. The folder must contain `data.yaml`, `train/`, `valid/`, and `test/`. Select a GPU runtime in **Runtime → Change runtime type**. Keep this notebook and all results private until the thesis team is ready to share them.

**Evaluation limit:** there is one physical slide per species. Test images are held-out photos from known slides; they are not an independent-slide test.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/pollens-thesis/Allergenic-PolLens-Classification.git'
REPO_REF = 'codex/ml-detection-local'  # Published experiment branch or later merged commit.
DATASET_SOURCE = Path('/content/drive/MyDrive/PolLens/datasets/Detection-2.zip')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PolLens/experiments/detection-v4-yolo11')
STAGE_DIR = Path('/content/pollens/detection-v4')
RESUME = False  # Set True only when continuing an interrupted run at OUTPUT_ROOT.

print('Dataset:', DATASET_SOURCE)
print('Results:', OUTPUT_ROOT)

## 1. Install the pinned trainer and check the GPU

Colab supplies its own CUDA-enabled PyTorch build. The project runner records the installed PyTorch, Python, and Ultralytics versions with the outputs.

In [ ]:
%pip -q install ultralytics==8.4.171

import torch, ultralytics
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime: Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'Ultralytics:', ultralytics.__version__)

## 2. Load the project code from GitHub

The public repository needs no GitHub token. The exact commit used for this run is recorded in Drive.

In [ ]:
import subprocess

REPO_DIR = Path('/content/PolLens')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
revision_file = OUTPUT_ROOT / 'code_revision.txt'
ref_to_fetch = revision_file.read_text().strip() if revision_file.exists() else REPO_REF
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1',
                'origin', ref_to_fetch], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
TRAINER = REPO_DIR / 'experiments/detection/train_local.py'
if not TRAINER.is_file():
    raise FileNotFoundError(f'Missing trainer at {TRAINER}. Check REPO_REF.')
REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if revision_file.exists() and revision_file.read_text().strip() != REVISION:
    raise ValueError('Could not load the original code revision for this output folder.')
revision_file.write_text(REVISION + '\n')
print('PolLens revision:', REVISION)

## 3. Locate Detection v4 and audit it

A Drive folder is used directly as the source. A ZIP is safely extracted to Colab local storage. The runner checks class names, YOLO labels, split counts and overlap, and a dataset fingerprint before training.

In [ ]:
import sys, zipfile

source = DATASET_SOURCE.expanduser()
if not source.exists():
    raise FileNotFoundError(f'Upload Detection v4 to Drive, then set DATASET_SOURCE: {source}')
if source.is_file():
    if not zipfile.is_zipfile(source):
        raise ValueError('DATASET_SOURCE must be a Detection v4 directory or ZIP file.')
    extracted = Path('/content/pollens/zip_extract')
    if not extracted.exists():
        extracted.mkdir(parents=True)
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                target = (extracted / member.filename).resolve()
                if not target.is_relative_to(extracted.resolve()):
                    raise ValueError(f'Unsafe ZIP member: {member.filename}')
            archive.extractall(extracted)
    candidates = [p.parent for p in extracted.rglob('data.yaml')
                  if all((p.parent / split / 'images').is_dir() and
                         (p.parent / split / 'labels').is_dir()
                         for split in ('train', 'valid', 'test'))]
    if len(candidates) != 1:
        raise ValueError(f'Expected one YOLO dataset in ZIP; found {len(candidates)}')
    dataset_root = candidates[0]
else:
    dataset_root = source

command = [sys.executable, str(TRAINER), '--dataset', str(dataset_root),
           '--output', str(OUTPUT_ROOT), '--audit-only']
subprocess.run(command, check=True)
print('Dataset root:', dataset_root)

## 4. Short GPU and memory check

This one-epoch run uses 10% of training images and writes only to Colab temporary storage. It is **not** a thesis result.

In [ ]:
SMOKE_OUTPUT = Path('/content/pollens/smoke-output')
smoke_command = [sys.executable, str(TRAINER), '--dataset', str(dataset_root),
                 '--output', str(SMOKE_OUTPUT), '--stage-dir', str(STAGE_DIR),
                 '--device', 'cuda', '--smoke']
subprocess.run(smoke_command, check=True)
print('Smoke check complete. Use the next cell for the full comparison.')

## 5. Train and evaluate YOLO11n and YOLO11s

Both models train on the same 292 original training images. Mild online augmentation applies only to training. The runner uses 640-pixel input, up to 80 epochs, patience 15, automatic CUDA batch sizing, mixed precision, and saved checkpoints. It evaluates validation and test once after both models finish. Keep `OUTPUT_ROOT` and `STAGE_DIR` the same when resuming.

In [ ]:
full_command = [sys.executable, str(TRAINER), '--dataset', str(dataset_root),
                '--output', str(OUTPUT_ROOT), '--stage-dir', str(STAGE_DIR),
                '--device', 'cuda']
if RESUME:
    full_command.append('--resume')
subprocess.run(full_command, check=True)
print('Comparison saved:', OUTPUT_ROOT / 'summary.csv')

## 6. Review results

The CSV compares precision, recall, F1, mAP@50, mAP@50–95, inference time, and fixed-confidence error counts. Keep the audit, metadata, package list, plots, checkpoints, and marked error examples with the thesis results. Do not use the test set to choose new settings.

In [ ]:
import csv
from IPython.display import display

summary = OUTPUT_ROOT / 'summary.csv'
if not summary.exists():
    raise FileNotFoundError(f'Full training is unfinished. Resume the previous cell: {summary}')
with summary.open(newline='') as stream:
    rows = list(csv.DictReader(stream))
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows:
        print(row)
print('Results folder:', OUTPUT_ROOT)